# 🩺 Diabetes Prediction using Logistic Regression

In this notebook, we build a diabetes prediction system using:

- Data cleaning and preprocessing
- Exploratory data inspection
- Feature scaling
- Logistic Regression (Scikit-Learn)
- Custom Logistic Regression implementation
- Model evaluation using classification metrics

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

diabetes_df = pd.read_csv('./diabetes.csv')
diabetes_df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 📦 Import Required Libraries & Load Dataset

In this step, we:

- Import essential Python libraries:
  - `pandas` for data manipulation
  - `numpy` for numerical operations
  - `matplotlib` for visualization
- Load the diabetes dataset using `pd.read_csv()`
- Display the first few rows to understand the dataset structure

In [2]:
diabetes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


## 🔍 Dataset Information

We use `.info()` to:

- Check total number of rows and columns
- Inspect data types of each feature
- Identify missing values
- Understand memory usage

This helps us understand the dataset structure before cleaning.

In [3]:
diabetes_df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


## 📊 Statistical Summary

Using `.describe()` we:

- View statistical metrics like mean, std, min, max
- Understand the distribution of numerical features
- Detect possible anomalies such as unrealistic values (e.g., zeros where not possible)

This step helps identify data quality issues.

In [4]:
print("Number of zeros or unknown values for each attributes.")
print(f"Glucose: {(diabetes_df['Glucose'] == 0).sum()}")
print(f"BloodPressure: {(diabetes_df['BloodPressure'] == 0).sum()}")
print(f"SkinThickness: {(diabetes_df['SkinThickness'] == 0).sum()}")
print(f"Insulin: {(diabetes_df['Insulin'] == 0).sum()}")
print(f"BMI: {(diabetes_df['BMI'] == 0).sum()}")

Number of zeros or unknown values for each attributes.
Glucose: 5
BloodPressure: 35
SkinThickness: 227
Insulin: 374
BMI: 11


## ⚠️ Checking for Invalid Zero Values

Certain medical attributes such as:

- Glucose
- Blood Pressure
- Skin Thickness
- Insulin
- BMI

cannot realistically be zero.

We count how many zero values exist in these columns to identify missing or invalid data.

In [5]:
columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
diabetes_df[columns] = diabetes_df[columns].replace(0, np.nan)
diabetes_df[columns] = diabetes_df[columns].fillna(diabetes_df[columns].median())

## 🧹 Handling Missing Values

To clean the dataset:

1. Replace invalid zero values with `NaN`
2. Fill missing values using the **median** of each column

Median is used because:
- It is robust to outliers
- Works well for skewed medical data

This ensures better model performance.

In [6]:
print("Number of zeros or unknown values for each attributes.")
print(f"Glucose: {(diabetes_df['Glucose'] == 0).sum()}")
print(f"BloodPressure: {(diabetes_df['BloodPressure'] == 0).sum()}")
print(f"SkinThickness: {(diabetes_df['SkinThickness'] == 0).sum()}")
print(f"Insulin: {(diabetes_df['Insulin'] == 0).sum()}")
print(f"BMI: {(diabetes_df['BMI'] == 0).sum()}")

Number of zeros or unknown values for each attributes.
Glucose: 0
BloodPressure: 0
SkinThickness: 0
Insulin: 0
BMI: 0


## ✅ Verifying Data Cleaning

After replacing and imputing missing values,  
we re-check the dataset to confirm that invalid zero values have been handled properly.

This ensures our preprocessing step worked correctly.

In [7]:
columns_to_select = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", 'DiabetesPedigreeFunction', 'Outcome']
cleaned_diabetes_df = diabetes_df.loc[:, columns_to_select]
cleaned_diabetes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Glucose                   768 non-null    float64
 1   BloodPressure             768 non-null    float64
 2   SkinThickness             768 non-null    float64
 3   Insulin                   768 non-null    float64
 4   BMI                       768 non-null    float64
 5   DiabetesPedigreeFunction  768 non-null    float64
 6   Outcome                   768 non-null    int64  
dtypes: float64(6), int64(1)
memory usage: 42.1 KB


In [8]:
cleaned_diabetes_df.head()

,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Outcome
0,148.0,72.0,35.0,125.0,33.6,0.627,1
1,85.0,66.0,29.0,125.0,26.6,0.351,0
2,183.0,64.0,29.0,125.0,23.3,0.672,1
3,89.0,66.0,23.0,94.0,28.1,0.167,0
4,137.0,40.0,35.0,168.0,43.1,2.288,1


## 👀 Preview Cleaned Dataset

We display the first few rows of the cleaned dataset  
to verify that preprocessing and feature selection were successful.

In [9]:
from sklearn.model_selection import train_test_split

X = cleaned_diabetes_df.drop('Outcome', axis=1)
Y = cleaned_diabetes_df['Outcome']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.8, random_state=42, stratify=Y)

## 🔀 Train-Test Split

We split the dataset into:

- 80% Training data
- 20% Testing data

Key points:
- `stratify=Y` ensures class balance in both sets
- `random_state=42` ensures reproducibility

This helps evaluate model performance on unseen data.

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 📏 Feature Scaling using StandardScaler

We apply feature scaling because:

- Logistic Regression is sensitive to feature magnitudes
- Scaling improves convergence speed
- Ensures fair contribution from all features

We:
- Fit the scaler on training data
- Transform both training and testing data

In [11]:
from sklearn.linear_model import LogisticRegression
logistic_model = LogisticRegression()

logistic_model.fit(X_train, Y_train)
logistic_model.score(X_test, Y_test)

0.7272727272727273

## 🤖 Logistic Regression (Scikit-Learn)

We:

- Initialize a Logistic Regression model
- Train it using the scaled training data
- Evaluate its performance using `.score()` on test data

This serves as our benchmark model.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_score, recall_score

Y_pred = logistic_model.predict(X_test)
conf_matrix = confusion_matrix(Y_test, Y_pred)
print("SciKit Learn Logistic Regression")
print(f"\nConfusion Matrix:\n{conf_matrix}")
print(f"Accuracy Score: {accuracy_score(Y_test, Y_pred)}")
print(f"F1 Score: {f1_score(Y_test, Y_pred)}")
print(f"Precision Score: {precision_score(Y_test, Y_pred)}")
print(f"Recall Score: {recall_score(Y_test, Y_pred)}")

SciKit Learn Logistic Regression
Confusion Matrix:
[[84 16]
 [26 28]]
Accuracy Score: 0.7272727272727273
F1 Score: 0.5714285714285714
Precision Score: 0.6363636363636364
Recall Score: 0.5185185185185185


In [ ]:
from logistic_regression import CustomLogisticRegression

custom_model = CustomLogisticRegression()
custom_model.fit(X_train, Y_train)
predictions = custom_model.predict(X_test)
conf_matrix = confusion_matrix(Y_test, predictions)
print("Custom Logistic Regression")
print(f"\nConfusion Matrix:\n{conf_matrix}")
print(f"Accuracy Score: {accuracy_score(Y_test, predictions)}")
print(f"F1 Score: {f1_score(Y_test, predictions)}")
print(f"Precision Score: {precision_score(Y_test, predictions)}")
print(f"Recall Score: {recall_score(Y_test, predictions)}")

Custom Logistic Regression
Confusion Matrix:
[[78 22]
 [21 33]]
Accuracy Score: 0.7207792207792207
F1 Score: 0.6055045871559633
Precision Score: 0.6
Recall Score: 0.6111111111111112


## 🛠 Custom Logistic Regression Implementation

Here we:

- Import our own implementation of Logistic Regression
- Train it on the same dataset
- Generate predictions
- Evaluate using the same metrics

This allows us to compare:
- Built-in Scikit-Learn model
- Custom implementation

and validate our understanding of the algorithm.

## 📈 Model Evaluation Metrics

We evaluate the model using:

- Confusion Matrix
- Accuracy Score
- Precision
- Recall
- F1 Score

These metrics give a deeper understanding of performance,  
especially for classification problems.